# 06. Self-checks

Three checks that the code does what it should. Run them after changing anything in `pqk100/`.

1. The light-cone simulator agrees with Qiskit's statevector on full circuits.
2. The lattice patches are connected and nested, and the gate layers are valid.
3. The hardware code path (layout, observables, parameter order) works end to end on FakeFez.

About 2 minutes. Nothing is sent to IBM.

In [1]:
import os, sys
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import networkx as nx
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeFez
from pqk100.lattice import patch, edge_layers
from pqk100.featuremap import FEATURE_MAPS, to_circuit, Op
from pqk100.lightcone import projections
from pqk100.hardware import build, submit, collect

def full_reference(n, ops, X):
    """<X>, <Y>, <Z> on every qubit from the full statevector."""
    qc, x = to_circuit(n, ops)
    ref = np.zeros((len(X), 3, n))
    for s, row in enumerate(X):
        sv = Statevector(qc.assign_parameters({x: row}))
        for a, P in enumerate("XYZ"):
            for k in range(n):
                lab = ["I"] * n; lab[n - 1 - k] = P
                ref[s, a, k] = sv.expectation_value(SparsePauliOp("".join(lab))).real
    return ref

project folder: D:\Active Research Projects\PQK100


## 6.1 Light-cone simulator against Qiskit

In [2]:
rng = np.random.default_rng(1)
n = 16
phys, edges = patch(FakeFez(), n)
X = rng.normal(size=(3, n))
for name, fm in FEATURE_MAPS.items():
    for reps in (1, 2, 3):
        ops = fm(n, edges, reps=reps)
        got, sizes = projections(n, ops, X)
        assert sizes.max() < n, "cone covers the whole patch; the check would be trivial"
        err = np.abs(got - full_reference(n, ops, X)).max()
        print(f"{name:6s} reps={reps} max cone={sizes.max():2d} max |diff|={err:.2e}")
        assert err < 1e-10

# every gate type, including ones the current maps do not use
rng = np.random.default_rng(2)
n = 5
ops = [Op("h", (q,)) for q in range(n)]
ops += [Op("rx", (0,), 0.7, (0,)), Op("ry", (1,), 1.3, (1,)), Op("rz", (2,), 0.4, ())]
ops += [Op("rxx", (0, 1), 0.9, (0, 1)), Op("ryy", (1, 2), 0.5, (2,)), Op("rzz", (2, 3), 1.1, (3, 2)), Op("cz", (3, 4))]
ops += [Op("ry", (q,), 1.0, (q,)) for q in range(n)] + [Op("rxx", (4, 3), 0.3, (4,)), Op("ryy", (2, 1), 0.8, ())]
X = rng.normal(size=(3, n))
err = np.abs(projections(n, ops, X)[0] - full_reference(n, ops, X)).max()
print(f"mixed gate set max |diff|={err:.2e}")
assert err < 1e-10
print("passed")

zz     reps=1 max cone= 3 max |diff|=1.80e-13
zz     reps=2 max cone= 5 max |diff|=1.27e-14
zz     reps=3 max cone= 7 max |diff|=1.07e-14
ry_cz  reps=1 max cone= 3 max |diff|=1.63e-14
ry_cz  reps=2 max cone= 5 max |diff|=9.33e-15
ry_cz  reps=3 max cone= 7 max |diff|=1.04e-14
mixed gate set max |diff|=4.44e-16
passed


## 6.2 Lattice patches

In [3]:
b = FakeFez(); prev = None
for n in (156, 127, 100, 60, 20):
    phys, edges = patch(b, n)
    g = nx.Graph(); g.add_nodes_from(range(n)); g.add_edges_from(edges)
    assert len(phys) == n and nx.is_connected(g)
    assert prev is None or set(phys) <= prev
    L = edge_layers(n, edges)
    assert len(L) <= 3 and sorted(e for l in L for e in l) == sorted(edges)
    assert all(len({q for e in l for q in e}) == 2 * len(l) for l in L)
    prev = set(phys)
    print(f"n={n:3d} edges={len(edges):3d} max degree={max(dict(g.degree).values())}")
print("passed")

n=156 edges=176 max degree=3
n=127 edges=128 max degree=3
n=100 edges= 99 max degree=3
n= 60 edges= 59 max degree=3
n= 20 edges= 19 max degree=2
passed


## 6.3 Hardware code path on FakeFez

The same build, submit and collect functions as on ibm_fez, run on the local noise model.
A wrong parameter order or qubit layout would destroy the agreement with the exact values.

In [4]:
backend = FakeFez()
n, reps, n_samples, shots = 10, 2, 12, 4000
rng = np.random.default_rng(3)
phys, edges = patch(backend, n)
ang = rng.normal(size=(n_samples, n)) * 0.6
for name, fm in FEATURE_MAPS.items():
    ops = fm(n, edges, reps=reps)
    exact, _ = projections(n, ops, ang)
    isa, cols, obs = build(n, ops, phys, backend)
    jobs = submit(isa, cols, obs, ang, backend, shots=shots, chunk=6, outdir="cache/selfcheck", tag=f"localtest_{name}")
    got, std = collect(jobs, n)
    assert got.shape == exact.shape == (n_samples, 3, n)
    r = np.corrcoef(got.ravel(), exact.ravel())[0, 1]
    slope = (got * exact).sum() / (exact ** 2).sum()
    print(f"{name:6s} n={n} reps={reps}: corr with exact = {r:.4f}, slope = {slope:.3f}, median stderr = {np.median(std):.3f}")
    assert r > 0.97 and 0.8 < slope < 1.02
    wrong, _ = projections(n, ops, ang[:, ::-1])
    assert np.corrcoef(got.ravel(), wrong.ravel())[0, 1] < 0.9
print("passed")

zz     n=10 reps=2: corr with exact = 0.9987, slope = 0.943, median stderr = 0.015
ry_cz  n=10 reps=2: corr with exact = 0.9984, slope = 0.963, median stderr = 0.015
passed
